# Notebook 01: Thu Thập & Chuẩn Hóa Dữ Liệu Chất Lượng Không Khí & Khí Tượng Hà Nội

- **Dự án:** Phân tích mức độ ô nhiễm không khí theo chuỗi thời gian tại Hà Nội (INFO3020 - Khoa học Dữ liệu)
- **Mục tiêu:** Thu thập dữ liệu thô ô nhiễm không khí từ OpenAQ S3 public archive (location_id=2178) và dữ liệu khí tượng tái phân tích ECMWF ERA5 từ Open-Meteo API trong giai đoạn 2023–2024, bảo toàn tính bất biến của tệp thô trong `data/raw/`, và ánh xạ chuẩn hóa sang **Canonical Data Schema** theo `docs/data_dictionary.md`.
- **Căn cứ kỹ thuật:** Milestone 1, GitHub Issue #3 & #19, và các quy tắc quản trị dữ liệu chuỗi thời gian trong `.agents/rules/data.md`.
- **Sản phẩm bàn giao:**
  - Tệp dữ liệu thô: `data/raw/openaq_raw_2023_2024.parquet`, `data/raw/open_meteo_raw_2023_2024.json`
  - Tệp canonical trung gian: `data/interim/air_quality_canonical.parquet`, `data/interim/weather_canonical.parquet`
  - Hồ sơ xuất xứ: `data/raw/metadata.json`


In [1]:
import sys
import os
import json
import logging
from pathlib import Path
import pandas as pd
import numpy as np

# Cố định cấu hình hiển thị pandas và logging
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Đảm bảo import được module từ thư mục gốc src/
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_collection import (
    OpenAQAdapter,
    OpenMeteoAdapter,
    HANOI_BBOX,
    CANONICAL_TIMEZONE,
    DEFAULT_STATION_CODE,
    DEFAULT_LOCATION_NAME,
    compute_file_sha256
)

print("Environment setup completed successfully!")


Environment setup completed successfully!


In [2]:
# Cố định cấu hình thời gian và đường dẫn
STUDY_START_DATE = '2023-01-01'
STUDY_END_DATE = '2024-12-31'
EXPECTED_TOTAL_HOURS = 17544  # 8760 (2023) + 8784 (2024 leap year)

RAW_DIR = project_root / 'data' / 'raw'
INTERIM_DIR = project_root / 'data' / 'interim'
METADATA_PATH = RAW_DIR / 'metadata.json'

print(f"Study Window: {STUDY_START_DATE} -> {STUDY_END_DATE}")
print(f"Timezone: {CANONICAL_TIMEZONE}")
print(f"Raw Storage Directory: {RAW_DIR}")
print(f"Interim Storage Directory: {INTERIM_DIR}")


Study Window: 2023-01-01 -> 2024-12-31
Timezone: Asia/Ho_Chi_Minh
Raw Storage Directory: D:\air-pollution-analysis\data\raw
Interim Storage Directory: D:\air-pollution-analysis\data\interim


## 1. Thu Thập Dữ Liệu Thô Chất Lượng Không Khí (OpenAQ S3 Archive)

Theo quyết định nguồn tại Issue #19, trạm quan trắc mặt đất của Đại sứ quán Hoa Kỳ tại Hà Nội được truy xuất thông qua kho lưu trữ công khai AWS S3 của OpenAQ:
`https://openaq-data-archive.s3.amazonaws.com/records/csv.gz/locationid=2178/`.
Dữ liệu thô tải về được lưu nguyên trạng vào `data/raw/openaq_raw_2023_2024.parquet` và được bảo vệ bất biến.


In [3]:
openaq_adapter = OpenAQAdapter(
    location_id=2178,
    station_id=DEFAULT_STATION_CODE,
    location_name=DEFAULT_LOCATION_NAME,
    raw_dir=RAW_DIR,
    interim_dir=INTERIM_DIR
)

# Nạp hoặc tải dữ liệu thô (nếu đã có tệp thô, adapter tự động nạp từ cache bất biến)
df_openaq_raw, openaq_raw_path = openaq_adapter.fetch_raw_data(
    start_date=STUDY_START_DATE,
    end_date=STUDY_END_DATE
)

print(f"Tệp thô OpenAQ: {openaq_raw_path}")
print(f"Kích thước dữ liệu thô: {df_openaq_raw.shape[0]:,} dòng, {df_openaq_raw.shape[1]} cột")
print(f"Mã băm SHA-256: {compute_file_sha256(openaq_raw_path)}")
df_openaq_raw.head()


2026-09-29 13:14:38,804 [INFO] Tệp thô OpenAQ đã tồn tại tại D:\air-pollution-analysis\data\raw\openaq_raw_2023_2024.parquet. Đang nạp từ kho lưu trữ bất biến...


Tệp thô OpenAQ: D:\air-pollution-analysis\data\raw\openaq_raw_2023_2024.parquet
Kích thước dữ liệu thô: 116,906 dòng, 9 cột
Mã băm SHA-256: 7286cb3da40488e27903a0daf314a538cf4a94163b27886e5c0d193b5783612a


,location_id,sensors_id,location,datetime,lat,lon,parameter,units,value
0,2178,3919,Del Norte-2178,2023-01-01T01:00:00-07:00,35.1353,-106.584702,pm10,µg/m³,45.0
1,2178,3919,Del Norte-2178,2023-01-01T02:00:00-07:00,35.1353,-106.584702,pm10,µg/m³,42.0
2,2178,3919,Del Norte-2178,2023-01-01T03:00:00-07:00,35.1353,-106.584702,pm10,µg/m³,26.0
3,2178,3919,Del Norte-2178,2023-01-01T04:00:00-07:00,35.1353,-106.584702,pm10,µg/m³,32.0
4,2178,3919,Del Norte-2178,2023-01-01T05:00:00-07:00,35.1353,-106.584702,pm10,µg/m³,26.0


## 2. Thu Thập Dữ Liệu Thô Khí Tượng Bề Mặt (Open-Meteo ERA5)

Theo quyết định tại Issue #19, dữ liệu khí tượng bề mặt được thu thập từ mô hình tái phân tích ECMWF ERA5 qua Open-Meteo Historical Archive API tại tọa độ vùng lõi Hà Nội (21.0285°N, 105.8542°E).
Payload JSON nguyên gốc từ API được lưu bất biến vào `data/raw/open_meteo_raw_2023_2024.json`.


In [4]:
weather_adapter = OpenMeteoAdapter(
    latitude=21.0285,
    longitude=105.8542,
    timezone=CANONICAL_TIMEZONE,
    raw_dir=RAW_DIR,
    interim_dir=INTERIM_DIR
)

weather_raw_json, weather_raw_path = weather_adapter.fetch_raw_data(
    start_date=STUDY_START_DATE,
    end_date=STUDY_END_DATE
)

print(f"Tệp thô Open-Meteo: {weather_raw_path}")
print(f"Số mốc giờ tải về: {len(weather_raw_json['hourly']['time']):,} mốc")
print(f"Mã băm SHA-256: {compute_file_sha256(weather_raw_path)}")
print("Các biến khí tượng tải về:", list(weather_raw_json['hourly'].keys()))


2026-09-29 13:14:38,850 [INFO] Tệp thô Open-Meteo đã tồn tại tại D:\air-pollution-analysis\data\raw\open_meteo_raw_2023_2024.json. Đang nạp từ kho lưu trữ bất biến...


Tệp thô Open-Meteo: D:\air-pollution-analysis\data\raw\open_meteo_raw_2023_2024.json
Số mốc giờ tải về: 17,544 mốc
Mã băm SHA-256: 7f12b53c60c83b172f1bd1b0723aaee9d5e4c3d2e17e65765aa9e108cc59d8a4
Các biến khí tượng tải về: ['time', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'wind_direction_10m', 'precipitation', 'surface_pressure']


## 3. Kiểm Tra Tính Toàn Vẹn Của Dữ Liệu Thô (Raw Verification)

Kiểm tra định dạng, kiểu dữ liệu, các tham số hiện diện và dải thời gian thực tế của cả hai nguồn dữ liệu thô trước khi chuyển đổi.


In [5]:
print("=== KIỂM TRA DỮ LIỆU THÔ OPENAQ ===")
print(df_openaq_raw.info())
print("\nDanh mục các chất ô nhiễm trong tệp thô OpenAQ:")
print(df_openaq_raw['parameter'].value_counts())

print("\n=== KIỂM TRA DỮ LIỆU THÔ OPEN-METEO ===")
sample_hourly = {k: v[:5] for k, v in weather_raw_json['hourly'].items()}
print(json.dumps(sample_hourly, indent=2))


=== KIỂM TRA DỮ LIỆU THÔ OPENAQ ===
<class 'pandas.DataFrame'>
RangeIndex: 116906 entries, 0 to 116905
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   location_id  116906 non-null  int64  
 1   sensors_id   116906 non-null  int64  
 2   location     116906 non-null  str    
 3   datetime     116906 non-null  str    
 4   lat          116906 non-null  float64
 5   lon          116906 non-null  float64
 6   parameter    116906 non-null  str    
 7   units        116906 non-null  str    
 8   value        116906 non-null  float64
dtypes: float64(3), int64(2), str(4)
memory usage: 13.1 MB
None

Danh mục các chất ô nhiễm trong tệp thô OpenAQ:
parameter
o3      15571
pm10    15442
so2     15281
co      14854
no2     14500
pm25    14400
no      13429
nox     13429
Name: count, dtype: int64

=== KIỂM TRA DỮ LIỆU THÔ OPEN-METEO ===
{
  "time": [
    "2023-01-01T00:00",
    "2023-01-01T01:00",
    "2023-01-01T02:00",
    

## 4. Chuẩn Hóa Sang Canonical Schema (Canonical Transformation)

Ánh xạ dữ liệu nguồn sang Canonical Data Schema trung lập (`docs/data_dictionary.md`):
1. **Chất lượng không khí:**
   - Parse datetime sang múi giờ chuẩn `Asia/Ho_Chi_Minh` (UTC+7).
   - Pivot `pm25` và `pm10` thành các cột độc lập.
   - Xử lý giá trị âm và missing markers (`-999`, `-9999`) thành `NaN`.
   - Giữ nguyên giá trị `0.0` nếu hợp lệ.
2. **Khí tượng bề mặt:**
   - Chuẩn hóa 6 trường: `temperature`, `relative_humidity`, `wind_speed`, `wind_direction`, `precipitation`, `surface_pressure`.


In [6]:
# Chuẩn hóa dữ liệu chất lượng không khí
df_air_canonical = openaq_adapter.to_canonical(df_openaq_raw)
print(f"Dữ liệu Canonical Không khí: {df_air_canonical.shape[0]:,} dòng, {df_air_canonical.shape[1]} cột")
display(df_air_canonical.head())

# Chuẩn hóa dữ liệu khí tượng
df_weather_canonical = weather_adapter.to_canonical(weather_raw_json)
print(f"\nDữ liệu Canonical Khí tượng: {df_weather_canonical.shape[0]:,} dòng, {df_weather_canonical.shape[1]} cột")
display(df_weather_canonical.head())


Dữ liệu Canonical Không khí: 15,501 dòng, 5 cột


parameter,timestamp,station_id,location,pm25,pm10
0,2023-01-01 15:00:00+07:00,VN001_HANOI_US_EMBASSY,US Diplomatic Post: Hanoi,NaN,45.0
1,2023-01-01 16:00:00+07:00,VN001_HANOI_US_EMBASSY,US Diplomatic Post: Hanoi,NaN,42.0
2,2023-01-01 17:00:00+07:00,VN001_HANOI_US_EMBASSY,US Diplomatic Post: Hanoi,NaN,26.0
3,2023-01-01 18:00:00+07:00,VN001_HANOI_US_EMBASSY,US Diplomatic Post: Hanoi,NaN,32.0
4,2023-01-01 19:00:00+07:00,VN001_HANOI_US_EMBASSY,US Diplomatic Post: Hanoi,NaN,26.0



Dữ liệu Canonical Khí tượng: 17,544 dòng, 7 cột


,timestamp,temperature,relative_humidity,wind_speed,wind_direction,precipitation,surface_pressure
0,2023-01-01 00:00:00+07:00,11.5,75,1.70,360,0.0,1022.4
1,2023-01-01 01:00:00+07:00,11.0,77,1.71,353,0.0,1022.0
2,2023-01-01 02:00:00+07:00,10.8,78,1.75,347,0.0,1021.5
3,2023-01-01 03:00:00+07:00,10.8,77,2.06,346,0.0,1021.4
4,2023-01-01 04:00:00+07:00,10.6,77,1.94,348,0.0,1021.5


## 5. Kiểm Chứng Địa Lý Hà Nội (Geographic Validation)

Kiểm tra đối chiếu tọa độ quan trắc thực tế với Bounding Box hành chính Hà Nội ($20.50^\circ	ext{N} 	o 21.60^\circ	ext{N}$, $105.30^\circ	ext{E} 	o 106.10^\circ	ext{E}$) theo quy định liêm chính học thuật.


In [7]:
is_within_hanoi, geo_details = openaq_adapter.validate_geographic_bounds(df_openaq_raw)
print("Kết quả kiểm chứng tọa độ trạm:")
print(json.dumps(geo_details, indent=2))

if not is_within_hanoi:
    print("\n[LƯU Ý LIÊM CHÍNH DỮ LIỆU]:")
    print(f"Tọa độ thực tế trạm ({geo_details.get('latitude')}, {geo_details.get('longitude')}) "
          f"nằm ngoài Bounding Box Hà Nội. Ghi nhận minh bạch hiện trạng này vào hồ sơ kiểm toán chất lượng.")
else:
    print("\nXác nhận trạm nằm trong ranh giới địa lý Hà Nội.")


Kết quả kiểm chứng tọa độ trạm:
{
  "latitude": 35.1353,
  "longitude": -106.584702,
  "bounding_box": {
    "lat_min": 20.5,
    "lat_max": 21.6,
    "lon_min": 105.3,
    "lon_max": 106.1
  },
  "is_within_hanoi": false
}

[LƯU Ý LIÊM CHÍNH DỮ LIỆU]:
Tọa độ thực tế trạm (35.1353, -106.584702) nằm ngoài Bounding Box Hà Nội. Ghi nhận minh bạch hiện trạng này vào hồ sơ kiểm toán chất lượng.


## 6. Kiểm Toán Tính Toàn Vẹn & Ràng Buộc Dữ Liệu (Integrity Checks)

Thực hiện các kiểm toán kỹ thuật cốt lõi:
- Đảm bảo timestamp được sắp xếp tuyến tính và duy nhất trên từng khóa quan sát `(station_id, timestamp)`.
- Đo lường số lượng bản ghi hợp lệ, số lượng khuyết thiếu và tỷ lệ missing rate.
- Kiểm tra các giới hạn vật lý tự nhiên (không có nồng độ âm).


In [8]:
# Kiểm tra tính duy nhất timestamp
assert df_air_canonical['timestamp'].is_unique, "Lỗi: Timestamp không duy nhất trong tập không khí!"
assert df_weather_canonical['timestamp'].is_unique, "Lỗi: Timestamp không duy nhất trong tập khí tượng!"

# Kiểm tra sắp xếp tuyến tính
assert df_air_canonical['timestamp'].is_monotonic_increasing, "Lỗi: Timestamp không tăng dần!"
assert df_weather_canonical['timestamp'].is_monotonic_increasing, "Lỗi: Timestamp thời tiết không tăng dần!"

# Kiểm tra số lượng giờ thời tiết chuẩn xác
assert len(df_weather_canonical) == EXPECTED_TOTAL_HOURS, f"Lỗi: Số giờ thời tiết {len(df_weather_canonical)} != {EXPECTED_TOTAL_HOURS}"

# Kiểm tra tỷ lệ missing
air_missing = df_air_canonical[['pm25', 'pm10']].isna().sum()
air_missing_pct = (df_air_canonical[['pm25', 'pm10']].isna().mean() * 100).round(2)
weather_missing = df_weather_canonical.drop(columns=['timestamp']).isna().sum()

print("=== KIỂM TOÁN KHUYẾT THIẾU TẬP KHÔNG KHÍ ===")
for col in ['pm25', 'pm10']:
    print(f"{col}: {df_air_canonical[col].notna().sum():,} quan sát hợp lệ | {air_missing[col]:,} khuyết ({air_missing_pct[col]}%)")

print("\n=== KIỂM TOÁN KHUYẾT THIẾU TẬP KHÍ TƯỢNG ===")
print(weather_missing)

# Kiểm tra giá trị âm
neg_pm25 = (df_air_canonical['pm25'] < 0).sum()
neg_pm10 = (df_air_canonical['pm10'] < 0).sum()
assert neg_pm25 == 0, "Phát hiện giá trị PM2.5 âm chưa được xử lý!"
assert neg_pm10 == 0, "Phát hiện giá trị PM10 âm chưa được xử lý!"
print("\nXác nhận: 100% giá trị nồng độ âm đã được chuyển đổi thành NaN hợp lệ.")


=== KIỂM TOÁN KHUYẾT THIẾU TẬP KHÔNG KHÍ ===
pm25: 14,400 quan sát hợp lệ | 1,101 khuyết (7.1%)
pm10: 15,442 quan sát hợp lệ | 59 khuyết (0.38%)

=== KIỂM TOÁN KHUYẾT THIẾU TẬP KHÍ TƯỢNG ===
temperature          0
relative_humidity    0
wind_speed           0
wind_direction       0
precipitation        0
surface_pressure     0
dtype: int64

Xác nhận: 100% giá trị nồng độ âm đã được chuyển đổi thành NaN hợp lệ.


## 7. Lưu Trữ Tệp Trung Gian & Cập Nhật Hồ Sơ Xuất Xứ (Interim Export & Provenance)

Lưu các bảng dữ liệu canonical trung gian vào `data/interim/` ở định dạng Snappy Parquet phục vụ khâu kiểm toán chất lượng (Issue #5) và đối chiếu với `data/raw/metadata.json`.


In [9]:
air_interim_path = INTERIM_DIR / 'air_quality_canonical.parquet'
weather_interim_path = INTERIM_DIR / 'weather_canonical.parquet'

df_air_canonical.to_parquet(air_interim_path, index=False, compression='snappy')
df_weather_canonical.to_parquet(weather_interim_path, index=False, compression='snappy')

print(f"Đã xuất bản tệp canonical không khí: {air_interim_path}")
print(f"Đã xuất bản tệp canonical khí tượng: {weather_interim_path}")

# Hiển thị thông tin hồ sơ provenance từ metadata.json
if METADATA_PATH.exists():
    with open(METADATA_PATH, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print("\n=== THÔNG TIN PROVENANCE TRONG METADATA.JSON ===")
    print("Schema Version:", meta.get('schema_version'))
    print("Target City:", meta.get('target_city'))
    print("Study Window:", meta.get('temporal_coverage', {}).get('study_window_start'), "->", meta.get('temporal_coverage', {}).get('study_window_end'))
    if 'collection_pipeline_execution' in meta:
        print("Pipeline Status:", meta['collection_pipeline_execution'].get('status'))
        print("Executed Timestamp:", meta['collection_pipeline_execution'].get('execution_timestamp_utc'))

print("\nHoàn tất Notebook 01 thành công 100%!")


Đã xuất bản tệp canonical không khí: D:\air-pollution-analysis\data\interim\air_quality_canonical.parquet


Đã xuất bản tệp canonical khí tượng: D:\air-pollution-analysis\data\interim\weather_canonical.parquet

=== THÔNG TIN PROVENANCE TRONG METADATA.JSON ===
Schema Version: 1.2.0
Target City: Hanoi, Vietnam
Study Window: 2023-01-01T00:00:00+07:00 -> 2024-12-31T23:00:00+07:00
Pipeline Status: completed
Executed Timestamp: 2026-09-29T06:13:01.324383+00:00

Hoàn tất Notebook 01 thành công 100%!
